In [8]:
# Ipython...
from __future__ import print_function, division
import matplotlib.pyplot as plt
from sympy.physics.vector import init_vprinting
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
get_ipython().run_line_magic('matplotlib', 'inline')
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
init_vprinting(use_latex='mathjax', pretty_print=False)
# Sympy
import numpy as np
import sympy as sp
from sympy import Matrix, symbols, simplify, Function, expand_trig, Symbol, diff
from sympy import cos, sin, transpose, pi
from sympy import latex, python
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, inertia
# Yams sympy
from welib.yams.yams_sympy       import YAMSRigidBody, YAMSInertialBody, YAMSFlexibleBody
from welib.yams.yams_sympy_model import YAMSModel


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# -- Degrees of freedom
time = dynamicsymbols._t
theta, psi = dynamicsymbols('theta, psi')
thetad, psid = dynamicsymbols('thetad, psid')

# --- Reference frame
ref = YAMSInertialBody('E') 
twr = YAMSFlexibleBody('T', nq=1, directions=['x'], orderMM=1, orderH=1, predefined_kind='twr-z') 
nac = YAMSRigidBody('N', rho_G = [Symbol('x_NG'),0,Symbol('z_NG')], J_form='diag') # full, diag, cross
#nac
bld = YAMSRigidBody('B', rho_G = [0,0,Symbol('z_BG')], J_form='diag') # full, diag, cross
bodies=[twr, nac, bld]
Ltwr = twr.L
# --- Connections
ref.connectTo(twr, type='Rigid', rel_pos=(0, 0, 0))
twr.connectToTip(nac, type='Joint', rel_pos=(0,0,Ltwr), rot_type_elastic = 'Body', doSubs=True)
nac.connectTo(bld, type='Joint', rel_pos=(Symbol('x_NB'), 0, Symbol('z_NB')), rot_type='Body', rot_amounts=(psi,0,0), rot_order='XYZ')



In [10]:
mySubs = []
mySubs += [(twr.ucList[0],0)]
mySubs += [(twr.vcList[0],1)]
mySubs

[(u_xT1c, 0), (v_yT1c, 1)]

In [11]:
twr.bodyMassMatrix(form='symbolic')
twr.bodyMassMatrix()

Matrix([
[  M_T,      0,   0,      0, M_T15,     0,  M_T17],
[    0,    M_T,   0, -M_T15,     0,     0,      0],
[    0,      0, M_T,      0,     0,     0,      0],
[    0, -M_T15,   0,  J_Txx,     0,     0,      0],
[M_T15,      0,   0,      0, J_Tyy,     0,  M_T57],
[    0,      0,   0,      0,     0, J_Tzz,      0],
[M_T17,      0,   0,      0, M_T57,     0, GM_T11]])

Matrix([
[       M_T,         0,   0,         0, -M_d^0_T_z,        0, C_t^0_T_1x],
[         0,       M_T,   0, M_d^0_T_z,          0,        0,          0],
[         0,         0, M_T,         0,          0,        0,          0],
[         0, M_d^0_T_z,   0,  J^0_T_xx,          0,        0,          0],
[-M_d^0_T_z,         0,   0,         0,   J^0_T_yy,        0, C_r^0_T_1y],
[         0,         0,   0,         0,          0, J^0_T_zz,          0],
[C_t^0_T_1x,         0,   0,         0, C_r^0_T_1y,        0, M_e^0_T_11]])

In [12]:
# --- DOFs and kinetic equations
coordinates = [twr.q[0], psi]
speeds      = [twr.qd[0], psid]
kdeqsSubs = []
kdeqsSubs += [(twr.qd[0], twr.qdot[0])]
kdeqsSubs += [(psid, sp.diff(psi, time))]
print('coordinates:', coordinates)
print('kdeqsSubs  :', kdeqsSubs)

coordinates: [q_T1(t), psi(t)]
kdeqsSubs  : [(qd_T1(t), Derivative(q_T1(t), t)), (psid(t), Derivative(psi(t), t))]


In [18]:
# --- Loads
gravity=Symbol('g')
body_loads = []
for bdy in bodies:
    body_loads  += [(bdy, (bdy.masscenter, -bdy.mass * gravity * ref.frame.z))]  

    
model = YAMSModel(name='NT', ref=ref, bodies=bodies, body_loads=body_loads, 
                  coordinates=coordinates, speeds=speeds, kdeqsSubs=kdeqsSubs, g_vect=-gravity*ref.frame.z)
model.kaneEquations(Mform='symbolic')
extraSubs  = mySubs

EOM=model.to_EOM(extraSubs=extraSubs)
EOM.mass_forcing_form()

In [19]:
print('Mass matrix:')
EOM.M
print('Forcing term:')
EOM.F

Mass matrix:


Matrix([
[GM_T11 + J_yy_B*cos(psi)**2 + J_yy_N + J_zz_B*sin(psi)**2 + M_B*x_NB**2 + M_B*z_BG**2*cos(psi)**2 + 2*M_B*z_BG*z_NB*cos(psi) + M_B*z_NB**2 + M_N*x_NG**2 + M_N*z_NG**2, M_B*x_NB*z_BG*sin(psi)],
[                                                                                                                                                M_B*x_NB*z_BG*sin(psi),   J_xx_B + M_B*z_BG**2]])

Forcing term:


Matrix([
[-D_e^0_T_11*q_T1' + 2*J_yy_B*sin(psi)*cos(psi)*psi'*q_T1' - 2*J_zz_B*sin(psi)*cos(psi)*psi'*q_T1' - K_e^0_T_11*q_T1 + M_B*g*x_NB*cos(q_T1) + M_B*g*z_BG*sin(q_T1)*cos(psi) + M_B*g*z_NB*sin(q_T1) - M_B*x_NB*z_BG*sin(psi)**2*cos(psi)*q_T1'**2 - M_B*x_NB*z_BG*cos(psi)**3*q_T1'**2 - M_B*x_NB*z_BG*cos(psi)*psi'**2 + M_B*x_NB*z_BG*cos(psi)*q_T1'**2 + 2*M_B*z_BG**2*sin(psi)*cos(psi)*psi'*q_T1' + 2*M_B*z_BG*z_NB*sin(psi)*psi'*q_T1' + M_N*g*x_NG*cos(q_T1) + M_N*g*z_NG*sin(q_T1)],
[                                                                                                                                                                                                                                                                                                       -J_yy_B*sin(psi)*cos(psi)*q_T1'**2 + J_zz_B*sin(psi)*cos(psi)*q_T1'**2 + M_B*g*z_BG*sin(psi)*cos(q_T1) - M_B*z_BG**2*sin(psi)*cos(psi)*q_T1'**2 - M_B*z_BG*z_NB*sin(psi)*q_T1'**2]])

In [15]:
M0,C0,K0,F0 = EOM.linearize(noVel=True)
M0
K0

Matrix([
[GM_T11 + J_yy_B*cos(psi)**2 + J_yy_N + J_zz_B*sin(psi)**2 + M_B*(x_NB**2 + z_BG**2*cos(psi)**2 + 2*z_BG*z_NB*cos(psi) + z_NB**2) + M_N*(x_NG**2 + z_NG**2), M_B*x_NB*z_BG*sin(psi)],
[                                                                                                                                    M_B*x_NB*z_BG*sin(psi),   J_xx_B + M_B*z_BG**2]])

Matrix([
[K_e^0_T_11 + M_B*g*x_NB*sin(q_T1) - M_B*g*z_BG*cos(psi)*cos(q_T1) - M_B*g*z_NB*cos(q_T1) + M_N*g*x_NG*sin(q_T1) - M_N*g*z_NG*cos(q_T1), M_B*g*z_BG*sin(psi)*sin(q_T1) + M_B*x_NB*z_BG*cos(psi)*Derivative(0, t) + (-2*J_yy_B*sin(psi)*cos(psi) + 2*J_zz_B*sin(psi)*cos(psi) + M_B*(-2*z_BG**2*sin(psi)*cos(psi) - 2*z_BG*z_NB*sin(psi)))*Derivative(0, t)],
[                                                                                                         M_B*g*z_BG*sin(psi)*sin(q_T1),                                                                                                                                          -M_B*g*z_BG*cos(psi)*cos(q_T1) + M_B*x_NB*z_BG*cos(psi)*Derivative(0, t)]])

In [16]:
#twrA.bodyMassMatrix(form='symbolic')
nac.bodyMassMatrix()

>>> bodyMassMatrix for rigid bodies is in Beta


Matrix([
[     M_N,         0,         0,                    0,                         M_N*z_NG,                    0],
[       0,       M_N,         0,            -M_N*z_NG,                                0,             M_N*x_NG],
[       0,         0,       M_N,                    0,                        -M_N*x_NG,                    0],
[       0, -M_N*z_NG,         0, J_xx_N + M_N*z_NG**2,                                0,       -M_N*x_NG*z_NG],
[M_N*z_NG,         0, -M_N*x_NG,                    0, J_yy_N + M_N*(x_NG**2 + z_NG**2),                    0],
[       0,  M_N*x_NG,         0,       -M_N*x_NG*z_NG,                                0, J_zz_N + M_N*x_NG**2]])